# Лабораторна робота №3

**Виконавець:** Сєров Едуард Олегович  
**Група:** К-26  
**Викладач:** Ляшко А. В  
**Варіант:** 19  
**Задача:** 1

---

**Умова:**  
За даними сайту https://www.numbeo.com/ знайти країну, в якій найдорожче та найдешевше харчування на 1 місяць (азіатське меню). Надати інформацію: де та на якому меню досягається.

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [2]:
HEADERS = {
    'User-Agent': (
        'Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:136.0) '
        'Gecko/20100101 Firefox/136.0'
    ),
    'Accept': (
        'text/html,application/xhtml+xml,application/xml;'
        'q=0.9,image/avif,image/webp,*/*;q=0.8'
    ),
    'Accept-Language': 'en-US,en;q=0.5',
}

BASE_URL = 'https://www.numbeo.com/food-prices/country_result.jsp'

In [3]:
r_main = requests.get('https://www.numbeo.com/food-prices/', headers=HEADERS)
print('Статус відповіді:', r_main.status_code)

soup_main = BeautifulSoup(r_main.text, 'html5lib')

countries = []
for sel in soup_main.find_all('select'):
    if sel.get('name') == 'country' or sel.get('id') == 'country':
        for opt in sel.find_all('option'):
            val = opt.get('value', '').strip()
            if val:
                countries.append(val)
        break

print(f'Знайдено країн: {len(countries)}')

Статус відповіді: 200
Знайдено країн: 234


In [4]:
def get_asian_monthly_usd(country: str) -> float | None | str:
    url = (
        f'{BASE_URL}'
        f'?country={country.replace(" ", "+")}'
        f'&displayCurrency=USD'
    )
    try:
        resp = requests.get(url, headers=HEADERS, timeout=20)

        if resp.status_code == 429:
            return 'RATE_LIMIT'

        if resp.status_code != 200:
            return None

        if 'Monthly recommended minimum' not in resp.text:
            return None

        soup = BeautifulSoup(resp.text, 'html5lib')
        monthly_count = 0

        for row in soup.find_all('tr'):
            cells = row.find_all('td')
            if not cells:
                continue
            label = cells[0].get_text(strip=True)
            if 'Monthly recommended minimum' in label:
                monthly_count += 1
                if monthly_count == 2:
                    val = cells[1].get_text(strip=True)
                    clean = ''.join(c for c in val if c.isdigit() or c == '.')
                    return float(clean) if clean else None

        return None

    except Exception as e:
        print(f'  Помилка для {country}: {e}')
        return None


print('Перевірка функції:')
for c in ['Ukraine', 'Japan', 'Switzerland']:
    cost = get_asian_monthly_usd(c)
    print(f'  {c:15s}: {cost} USD/міс.')

Перевірка функції:
  Ukraine        : 125.44 USD/міс.
  Japan          : 279.61 USD/міс.
  Switzerland    : 544.42 USD/міс.


In [5]:
results  = {}
rate_hit = False
total    = len(countries)

for i, country in enumerate(countries):
    cost = get_asian_monthly_usd(country)

    if cost == 'RATE_LIMIT':
        rate_hit = True
        print(f'\n>>> Rate limit на країні [{i + 1}/{total}]: {country}')
        print(f'>>> Зібрано до ліміту: {len(results)} країн')
        break

    results[country] = cost

    collected = sum(1 for v in results.values() if v is not None)
    if (i + 1) % 10 == 0:
        print(f'  [{i + 1:3d}/{total}]  з даними: {collected}'
              f'  | {country} -> {cost}')

if not rate_hit:
    collected = sum(1 for v in results.values() if v is not None)
    print(f'\nЗбір завершено. Країн з даними: {collected} / {len(results)}')

  [ 10/234]  з даними: 10  | Antigua And Barbuda -> 442.73
  [ 20/234]  з даними: 20  | Barbados -> 351.85
  [ 30/234]  з даними: 29  | Botswana -> 140.96
  [ 40/234]  з даними: 39  | Cape Verde -> 253.64

>>> Rate limit на країні [49/234]: Costa Rica
>>> Зібрано до ліміту: 48 країн


In [6]:
df = pd.DataFrame(
    [(k, v) for k, v in results.items() if v is not None],
    columns=['country', 'asian_menu_monthly_usd']
)

df['asian_menu_monthly_usd'] = pd.to_numeric(
    df['asian_menu_monthly_usd'], errors='coerce'
)
df.dropna(subset=['asian_menu_monthly_usd'], inplace=True)
df.sort_values('asian_menu_monthly_usd', ascending=False, inplace=True)
df.reset_index(drop=True, inplace=True)

print(f'Країн у датасеті: {len(df)}')
print()
display(df)

Країн у датасеті: 45



,country,asian_menu_monthly_usd
0,Bermuda,659.54
1,Cayman Islands,582.96
2,Bahamas,448.23
3,Antigua And Barbuda,442.73
4,Alderney,440.45
5,British Virgin Islands,406.55
6,Anguilla,390.70
7,Barbados,351.85
8,Austria,328.13
9,Aruba,311.36


In [7]:
print('Статистика — Asian Menu Monthly Cost (USD):')
display(df['asian_menu_monthly_usd'].describe().round(2).to_frame())

Статистика — Asian Menu Monthly Cost (USD):


,asian_menu_monthly_usd
count,45.00
mean,247.45
std,126.76
min,94.32
25%,153.81
50%,200.01
75%,301.77
max,659.54


In [8]:
row_max = df.iloc[0]
row_min = df.iloc[-1]

note = (f'(вибірка: {len(df)} країн — '
        f'повний збір обмежено rate limit сайту)')

print('=' * 65)
print(f'ВІДПОВІДЬ (Варіант 19)  {note}')
print()
print('НАЙДОРОЖЧЕ харчування на 1 місяць:')
print(f'  Країна  : {row_max["country"]}')
print(f'  Вартість: ${row_max["asian_menu_monthly_usd"]:.2f} USD/місяць')
print(f'  Меню    : Asian Menu (monthly recommended minimum amount)')
print()
print('НАЙДЕШЕВШЕ харчування на 1 місяць:')
print(f'  Країна  : {row_min["country"]}')
print(f'  Вартість: ${row_min["asian_menu_monthly_usd"]:.2f} USD/місяць')
print(f'  Меню    : Asian Menu (monthly recommended minimum amount)')
print('=' * 65)

ВІДПОВІДЬ (Варіант 19)  (вибірка: 45 країн — повний збір обмежено rate limit сайту)

НАЙДОРОЖЧЕ харчування на 1 місяць:
  Країна  : Bermuda
  Вартість: $659.54 USD/місяць
  Меню    : Asian Menu (monthly recommended minimum amount)

НАЙДЕШЕВШЕ харчування на 1 місяць:
  Країна  : Bangladesh
  Вартість: $94.32 USD/місяць
  Меню    : Asian Menu (monthly recommended minimum amount)


## Висновок

За даними сайту numbeo.com (розділ Food Prices, Asian Menu,
«monthly recommended minimum amount», у доларах США):

- **Найдорожче** азіатське харчування на місяць — **Bermuda** ($659.54 USD/міс.)

- **Найдешевше** азіатське харчування на місяць — **Bangladesh** ($94.32 USD/міс.)

Дані зібрано програмно за допомогою бібліотек `requests` та `beautifulsoup4`.
Для кожної країни зі списку надсилався GET-запит на сторінку
`country_result.jsp` з параметром `displayCurrency=USD`.
Значення Asian Menu визначалось як **другий** рядок
«Monthly recommended minimum amount» на сторінці країни.

**Примітка:** Сайт numbeo.com має обмеження безкоштовного плану
(rate limit — ~100 запитів/місяць). Аналіз виконано на вибірці
**45 країн** — усіх, дані яких вдалося отримати програмно
до досягнення ліміту (збір зупинився на країні Costa Rica).